# Experiments in grabbing data from Execution Rocks, storing it and retrieving it for the Kiosk

We OCR the screens on the LIRICOSs (sp?) and store it in the cloud using the free tier of an AI non-SQL DB. We then can retrieve the information for the kiosk. The OCR and storage happends on 'jeeves' a local server that can run quietly 24/7.



In [10]:
# Boilerplate starting points
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import CloudSecrets as cdbs

print("CloudSecrets module imported successfully.")
print(f"LOKI_URL:        {cdbs.CLD.LOKI_URL}")
print(f"LOKI_USER:       {cdbs.CLD.LOKI_USER}")
print(f"LOKI_TOKEN_WRTE: {cdbs.CLD.LOKI_TOKEN_WRTE}")
print(f"LOKI_TOKEN_READ: {cdbs.CLD.LOKI_TOKEN_READ}")

CloudSecrets module imported successfully.
LOKI_URL:        https://logs-prod-042.grafana.net
LOKI_USER:       1639785
LOKI_TOKEN_WRTE: glc_eyJvIjoiMTgwMzA4OCIsIm4iOiJzdGFjay0xNjgxOTk4LWFsbG95LWJ1b3lkYXRhLXB1c2gtdG9rZW4iLCJrIjoiUVoxSTFZMG1BUzczMzY5cFE3SkZySTFiIiwibSI6eyJyIjoicHJvZC11cy1lYXN0LTMifX0=
LOKI_TOKEN_READ: glc_eyJvIjoiMTgwMzA4OCIsIm4iOiJzdGFjay0xNjgxOTk4LWhsLXJlYWQtYnVveWRhdGEtcmVhZC10b2tlbi1raW9zayIsImsiOiIyUkEzMDY0NTFuMzQ3WUhjY3RLWVVmSFciLCJtIjp7InIiOiJwcm9kLXVzLWVhc3QtMyJ9fQ==


In [ ]:
# connection to Grafana Loki for log retrieval
import os
import time
import requests

LOKI_HOST = cdbs.CLD.LOKI_URL   # same host you push to
LOKI_USER = cdbs.CLD.LOKI_USER   # numeric instance ID, not the token
API_TOKEN = cdbs.CLD.LOKI_TOKEN_READ # must include logs:read scope

# this fails wuith a 404 because it doesn't push through the
# firewalls.  It is meant for internal testing.
"""
resp = requests.get(
    f"{LOKI_HOST}/ready",
    auth=(LOKI_USER, API_TOKEN),
    timeout=10,
)
print(resp.status_code, resp.text)  # 200 "ready" when healthy
"""

# This is a better end to end test
resp = requests.get(
    f"{LOKI_HOST}/loki/api/v1/labels",
    params={"start": start_ns, "end": end_ns},
    auth=(LOKI_USER, API_TOKEN),
    timeout=10,
)

resp.raise_for_status()
print(resp.json())  # {"status":"success","data":["__name__","job","pod",...]}
# should return {'status': 'success', 'data': ['device', 'location', 'service_name']}
# if all is well.

{'status': 'success', 'data': ['device', 'location', 'service_name']}


In [13]:
# Tests with time module
print(f"Time [local] result:   {time.time()}")  # Current time in seconds since the epoch (UTC)
print(f"Time [UTC] result:     {time.gmtime(time.time())}")
print(f"Time timezone:         {time.timezone/3600}")
print(f"Time human-readable:   {time.ctime()}")
print(f"Time daylight:         {time.daylight}")
print(f"Time_ns (1):           {time.clock_gettime_ns(time.CLOCK_REALTIME)}")
print(f"Time_ns (2):           {time.time_ns()}")
print(f"Time_ns (3):           {int(time.time() * 1e9)}")

Time [local] result:   1789915793.7020907
Time [UTC] result:     time.struct_time(tm_year=2026, tm_mon=9, tm_mday=20, tm_hour=14, tm_min=49, tm_sec=53, tm_wday=6, tm_yday=263, tm_isdst=0)
Time timezone:         5.0
Time human-readable:   Sun Sep 20 10:49:53 2026
Time daylight:         1
Time_ns (1):           1789915793702458145
Time_ns (2):           1789915793702504660
Time_ns (3):           1789915793702560256


In [14]:
# --- Configuration ---
# Defined above in the notebook

LOGQL_QUERY = '{service_name="unknown_service"} | json | logfmt | drop __error__, __error_details__'

# --- Time range: last 18 hours ---
end_ns = int(time.time_ns()  + int(20 * 60 * 1e9))  # put the end point 20 min in the future to account for clock skew
start_ns = end_ns - int(18 * 60 * 60 * 1e9)  # back up 36 hours

# --- Query Loki directly ---
url = f"{LOKI_HOST}/loki/api/v1/query_range"
params = {
    "query": LOGQL_QUERY,
    "start": start_ns,
    "end": end_ns,
    "limit": 5000,       # Loki caps results per request; paginate below if you hit this
    "direction": "forward",
}

resp = requests.get(url, params=params, auth=(LOKI_USER, API_TOKEN), timeout=30)
resp.raise_for_status()
data = resp.json()

# --- Flatten Loki streams into rows ---
rows = []
for stream in data.get("data", {}).get("result", []):
    labels = stream.get("stream", {})
    for ts_ns, line in stream.get("values", []):
        rows.append({
            "timestamp": pd.to_datetime(int(ts_ns), unit="ns", utc=True),
            "line": line,
            **labels,
        })

df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
print(f"Retrieved {len(df)} log lines")
df.head()


Retrieved 213 log lines


,timestamp,line,AirTemp_degC,AirTemp_degF,BaromPres_mB,BaromPres_mmHg,DewPoint_degC,DewPoint_degF,RelHum_perc,Source,...,WindSpeedAvg_mps,WindSpeedGst_kts,WindSpeedGst_mph,WindSpeedGst_mps,WindSpeedM24_kts,WindTimeM24,detected_level,device,location,service_name
0,2026-09-19 21:19:05.391134042+00:00,"{""WindSpeedAvg_kts"": 10.9, ""WindSpeedGst_kts"":...",19.0,66.2,1020.49,30.14,14.3,57.8,72.4,E R B - NOAA 44022,...,5.6,14.8,17.0,7.6,28.6,2026-09-19T06:04:00-04:00,unknown,NERACOOS_buoy,exrx,unknown_service
1,2026-09-19 21:19:20.019778424+00:00,"{""WindSpeedAvg_kts"": 3.5, ""WindSpeedGst_kts"": ...",18.7,65.7,1021.35,29.99,10.7,1.2,59.5,C S B - NOAA 44039,...,1.8,6.3,7.2,3.2,28.4,2026-09-19T02:04:00-04:00,unknown,NERACOOS_buoy,clis,unknown_service
2,2026-09-19 21:19:34.580940896+00:00,"{""WindSpeedAvg_kts"": 8.7, ""WindSpeedGst_kts"": ...",18.6,65.4,1021.1,30.15,13.2,55.8,69.9,W S B - NOAA 44040,...,4.5,13.2,15.2,6.8,26.9,2026-09-19T03:49:00-04:00,unknown,NERACOOS_buoy,wlis,unknown_service
3,2026-09-19 21:34:05.292484767+00:00,"{""WindSpeedAvg_kts"": 10.5, ""WindSpeedGst_kts"":...",19.1,66.4,1020.49,30.14,14.4,58.0,72.3,E R B - NOAA 44022,...,5.4,15.8,18.2,8.1,28.6,2026-09-19T06:04:00-04:00,unknown,NERACOOS_buoy,exrx,unknown_service
4,2026-09-19 21:34:19.809736564+00:00,"{""WindSpeedAvg_kts"": 3.5, ""WindSpeedGst_kts"": ...",18.7,65.7,1021.35,29.99,10.7,1.2,59.5,C S B - NOAA 44039,...,1.8,6.3,7.2,3.2,28.4,2026-09-19T02:04:00-04:00,unknown,NERACOOS_buoy,clis,unknown_service
